# 04 — 8-Cell Ablation Batch Eval (Kaggle T4)

Runs the full 8-cell ablation matrix and produces aggregated comparison tables.

## Output: `results/ablation_table.csv`

| Ablation | Recall@10 | MRR@10 | nDCG@10 | F1 | ROUGE-L | Citation F1 | Faithfulness |
|---|---|---|---|---|---|---|---|
| A1 Base RAG | ... | ... | ... | ... | ... | ... | ... |
| ... | ... | ... | ... | ... | ... | ... | ... |
| A5 Full FT | ... | ... | ... | ... | ... | ... | ... |

## 8-cell matrix

| ID | Embedding | Mode | Reranker | LLM | Isolates |
|---|---|---|---|---|---|
| A1 | base e5 | bm25 | none | Trendyol vanilla | baseline |
| A2 | base e5 | hybrid | none | Trendyol vanilla | retrieval mode |
| A3 | base e5 | hybrid | BGE pretrained | Trendyol vanilla | reranker presence |
| A4 | base e5 | hybrid | BGE-v2 FT | Trendyol vanilla | **reranker FT** (vs A3) |
| A5 | **e5-FT** | hybrid | **BGE-v2 FT** | **Trendyol-SFT** | **Full FT** |
| A5a | e5-FT | hybrid | BGE-v2 FT | Trendyol vanilla | **LLM SFT** (vs A5) |
| A5b | base e5 | hybrid | BGE-v2 FT | Trendyol-SFT | **Embed FT** (vs A5) |
| A5c | e5-FT | hybrid | BGE pretrained | Trendyol-SFT | Reranker FT (vs A5) |

## Time budget
- Per ablation: ~3 min model load + 122 questions × ~10 s ≈ 22 min
- Full 8-cell: ~3 hours (within Kaggle's 9h limit)
- Recommended order: A1 + A5 first (primary comparison), then the rest

## Execution
1. Run all A1–A5 + A5a/b/c cells sequentially via **Save & Run All**
2. Or run the critical cells (A1, A5) first as a minimum-evidence pass
3. The final cell aggregates results — missing ablations appear as NaN

---

**v2 (2026-05-22):** all ablations re-run on cleaned corpus (37,927 maddes, gold-isolation enforced) with  +  (listwise CE). Outputs to  to preserve v1 results for delta comparison.

In [ ]:
%%capture
# Kaggle env setup — paketler genelde preinstalled ama emin olalim
!pip install -q peft sentence-transformers rank_bm25 faiss-cpu rouge-score sacrebleu
# Kaggle Trendyol icin trust_remote_code, bnb 4-bit, accelerate
!pip install -q -U bitsandbytes accelerate


In [ ]:
import os, shutil, subprocess, json
from pathlib import Path

INPUT = Path('/kaggle/input/datasets/hasanemreusta/turkish-legal-rag-system')
assert (INPUT / 'src').exists() and (INPUT / 'data').exists(), f'src/+data/ bulunamadi: {INPUT}'

WORK = Path('/kaggle/working/turkish-legal-rag')
WORK.mkdir(exist_ok=True, parents=True)

if not (WORK / 'src').exists():
    shutil.copytree(INPUT / 'src', WORK / 'src')

# data/ hibrit kurulum: test_set ve models symlink, index_full birlestir
data_dir = WORK / 'data'
if data_dir.is_symlink():
    data_dir.unlink()
data_dir.mkdir(exist_ok=True)
for sub in ['test_set', 'models']:
    tgt = data_dir / sub
    if not tgt.exists():
        os.symlink(INPUT / 'data' / sub, tgt)

# index_full: tum base dosyalari + ft_index_patch'taki FT dosyalari birlestir
idx = data_dir / 'index_full'
idx.mkdir(exist_ok=True)
for f in (INPUT / 'data' / 'index_full').iterdir():
    link = idx / f.name
    if not link.exists():
        os.symlink(f, link)

# FT FAISS + embeddings'i ft_index_patch icinden bul ve symlink et
ft_patch_root = INPUT / 'ft_index_patch'
ft_files_found = []
if ft_patch_root.exists():
    for root, dirs, files in os.walk(ft_patch_root):
        for fname in files:
            if 'data_models_e5-large-tr-legal-v2' in fname and (fname.endswith('.index') or fname.endswith('.npy')):
                src = Path(root) / fname
                link = idx / fname
                if not link.exists():
                    os.symlink(src, link)
                ft_files_found.append(fname)
print(f'FT files linked: {ft_files_found}')

(WORK / 'results').mkdir(exist_ok=True)

os.chdir(WORK)
print('CWD:', os.getcwd())
print('src/:', sorted(p.name for p in (WORK/'src').iterdir()))
print('data/:', sorted(p.name for p in (WORK/'data').iterdir()))
print('Gold set:', (WORK/'data/test_set/dev_full.jsonl').exists())
print('index_full contents:')
for f in sorted((WORK/'data/index_full').iterdir()):
    print(f'  {f.name}')
print('Models:', sorted(p.name for p in (WORK/'data/models').iterdir()))

# Sanity: FT FAISS index orada mi?
ft_faiss = WORK/'data/index_full/faiss_data_models_e5-large-tr-legal-v2.index'
assert ft_faiss.exists(), f'FT FAISS bulunamadi: {ft_faiss}'
print('FT FAISS OK:', ft_faiss)

print('Pre-caching multilingual-e5-large...')
from sentence_transformers import SentenceTransformer
_m = SentenceTransformer('intfloat/multilingual-e5-large')
del _m
import gc, torch
gc.collect(); torch.cuda.empty_cache()
print('e5-large cached OK.')


In [ ]:
# Ortak ayarlar
GOLD = 'data/test_set/dev_full.jsonl'
INDEX = 'data/index_full'
EMBED_BASE = 'intfloat/multilingual-e5-large'
EMBED_FT = 'data/models/e5-large-tr-legal-v2'        # v2: cleaned-corpus FT
RERANKER_FT = 'data/models/bge-reranker-tr-legal-v3'  # v3: listwise softmax CE
RERANKER_PRE = 'BAAI/bge-reranker-v2-m3'
LLM_BASE = 'Trendyol/Trendyol-LLM-7B-chat-v4.1.0'
ADAPTER = 'data/models/llm_adapter'

# Per-ablation timeout: 2h ceiling. With max_new_tokens=256 patch, expect 25-45min/ablation.
ABLATION_TIMEOUT_SEC = 7200

# PATCH: run_eval.py'de max_new_tokens default 512 -> 256 (generation ~2x speedup)
import pathlib
_re = pathlib.Path('src/eval/run_eval.py')
_c = _re.read_text(encoding='utf-8')
_old = 'rag.llm.generate(SYSTEM_PROMPT, user_prompt)'
_new = 'rag.llm.generate(SYSTEM_PROMPT, user_prompt, max_new_tokens=256)'
if _old in _c:
    _re.write_text(_c.replace(_old, _new), encoding='utf-8')
    print('PATCH OK: max_new_tokens=256')
else:
    print('PATCH SKIP: already patched or pattern not found')

COMMON = [
    'python', '-u', '-m', 'src.eval.run_eval',
    '--test', GOLD,
    '--index-dir', INDEX,
    '--llm-backend', 'hf',
    '--llm-model', LLM_BASE,
    '--top-k', '5',
    '--candidate-k', '50',
]

def run_ablation(name, extra_args, out_root='results'):
    out_dir = f'{out_root}/{name}'
    cmd = COMMON + ['--out', out_dir] + extra_args
    print('', flush=True)
    print('=' * 60, flush=True)
    print(name, flush=True)
    print('=' * 60, flush=True)
    print('CMD:', ' '.join(cmd), flush=True)
    try:
        result = subprocess.run(cmd, capture_output=False, timeout=ABLATION_TIMEOUT_SEC)
    except subprocess.TimeoutExpired:
        print('!! ' + name + ' TIMEOUT (' + str(ABLATION_TIMEOUT_SEC) + 's asildi) - atlaniyor', flush=True)
        return None
    if result.returncode != 0:
        print('!! ' + name + ' BASARISIZ (returncode=' + str(result.returncode) + ') - devam', flush=True)
        return None
    summary_path = WORK / out_dir / 'summary.json'
    if summary_path.exists():
        s = json.loads(summary_path.read_text())
        print('OK ' + name + ' bitti.', flush=True)
        return s
    return None

print('Hazir. Cell sirasi: A1 -> A5 -> A5a -> A5b -> A5c -> A2 -> A3 -> A4 -> aggregate')


def run_ablation_v2(name, extra_args):
    """Wrapper writing to results_v2/ (preserve v1 results in results/)."""
    return run_ablation(name, extra_args, out_root='results_v2')


## A1 — Base RAG (BM25 + vanilla Trendyol)
Baseline. ~20 min.

In [ ]:
s_A1 = run_ablation_v2('A1', [
    '--mode', 'bm25',
    '--embed-model', EMBED_BASE,
])
s_A1

## A5 — Full FT (e5-FT + BGE-v2 FT + Trendyol-SFT)
Full Fine-tuned RAG — primary comparison point against A1.

In [ ]:
s_A5 = run_ablation_v2('A5', [
    '--mode', 'hybrid',
    '--embed-model', EMBED_FT,
    '--reranker', 'v2',
    '--reranker-dir', RERANKER_FT,
    '--adapter-path', ADAPTER,
])
s_A5

## A5a — A5 − LLM SFT (LoRA adapter çıkarıldı)
A5 vs A5a farkı = **LLM SFT izolasyonu**.

In [ ]:
s_A5a = run_ablation_v2('A5a', [
    '--mode', 'hybrid',
    '--embed-model', EMBED_FT,
    '--reranker', 'v2',
    '--reranker-dir', RERANKER_FT,
])
s_A5a

## A5b — A5 − Embed FT (base e5 kullanıldı)
A5 vs A5b farkı = **Embed FT izolasyonu**.

In [ ]:
s_A5b = run_ablation_v2('A5b', [
    '--mode', 'hybrid',
    '--embed-model', EMBED_BASE,
    '--reranker', 'v2',
    '--reranker-dir', RERANKER_FT,
    '--adapter-path', ADAPTER,
])
s_A5b

## A5c — A5 − Reranker FT (BGE pretrained)
A5 vs A5c farkı = Reranker FT izolasyonu (ikinci kanıt).

In [ ]:
s_A5c = run_ablation_v2('A5c', [
    '--mode', 'hybrid',
    '--embed-model', EMBED_FT,
    '--reranker', 'pretrained',
    '--reranker-dir', RERANKER_PRE,
    '--adapter-path', ADAPTER,
])
s_A5c

## A2 — +Hybrid (BM25 + dense base e5)

In [ ]:
s_A2 = run_ablation_v2('A2', [
    '--mode', 'hybrid',
    '--embed-model', EMBED_BASE,
])
s_A2

## A3 — +Reranker (pretrained BGE)

In [ ]:
s_A3 = run_ablation_v2('A3', [
    '--mode', 'hybrid',
    '--embed-model', EMBED_BASE,
    '--reranker', 'pretrained',
    '--reranker-dir', RERANKER_PRE,
])
s_A3

## A4 — +Reranker FT (BGE-v2 fine-tuned)
A4 vs A3 farkı = **reranker FT izolasyonu** (rapor için kritik).

In [ ]:
s_A4 = run_ablation_v2('A4', [
    '--mode', 'hybrid',
    '--embed-model', EMBED_BASE,
    '--reranker', 'v2',
    '--reranker-dir', RERANKER_FT,
])
s_A4

## Aggregate — Tablo 1 üretimi

Tüm summary.json dosyalarını toplayıp tek bir karşılaştırma tablosu üret. Rapor için.

In [ ]:
import pandas as pd

ABLATIONS = ['A1', 'A2', 'A3', 'A4', 'A5', 'A5a', 'A5b', 'A5c']
RETR_COLS = ['recall@1', 'recall@3', 'recall@5', 'recall@10', 'mrr@10', 'ndcg@10']
ANS_COLS = ['em', 'f1', 'bleu', 'rouge_l']
CIT_COLS = ['cit_p', 'cit_r', 'cit_f1']

rows = []
for name in ABLATIONS:
    sp = WORK / f'results_v2/{name}/summary.json'
    if not sp.exists():
        rows.append({'ablation': name, 'status': 'MISSING'})
        continue
    s = json.loads(sp.read_text())
    row = {'ablation': name, 'status': 'OK', 'n': s.get('n_examples', '?')}
    # Flatten nested groups
    for k in RETR_COLS:
        v = s.get('retrieval', {}).get(k)
        row[k] = round(v, 4) if isinstance(v, (int, float)) else None
    for k in ANS_COLS:
        v = s.get('answer', {}).get(k)
        row[k] = round(v, 4) if isinstance(v, (int, float)) else None
    for k in CIT_COLS:
        v = s.get('citation', {}).get(k)
        row[k] = round(v, 4) if isinstance(v, (int, float)) else None
    fm = s.get('faithfulness_mean')
    row['faith'] = round(fm, 4) if isinstance(fm, (int, float)) else None
    rows.append(row)

df = pd.DataFrame(rows)
df.to_csv(WORK / 'results_v2/ablation_table.csv', index=False)
print(df.to_string(index=False))
df


## Output download

Notebook commit edildikten sonra Output sekmesinden:
- `results/A1/per_example.jsonl` ... `results/A5c/per_example.jsonl` (her soru için detay)
- `results/A1/summary.json` ... (her ablation için ortalama)
- `results/ablation_table.csv` ⭐ (rapor için ana tablo)

## Sıradaki: Faithfulness judge (lokal, Anthropic Haiku)

Bu notebook'ta judge çağrısı yok (Anthropic API key Kaggle'da yok). Lokalde:

```powershell
python -m src.eval.run_eval --test data/test_set/dev_full.jsonl --judge `
  --in results/A5/per_example.jsonl --out results/A5_with_judge
```

veya direkt `judge.py`'ı per_example.jsonl üzerinde tek başına koş.